# SUPIR: базовое восстановление одного изображения

Минимальный запуск предобученной **SUPIR-v0Q** без LLaVA: описание изображения задаётся вручную. Ноутбук использует код этого репозитория и повторяет путь `create_SUPIR_model` → `PIL2Tensor` → `batchify_sample` → `Tensor2PIL` из `test.py`.

Нужна Linux-машина с CUDA, локальными весами SDXL и SUPIR. На ней из корня репозитория выполните `uv venv --python 3.10` и `uv pip install --python .venv/bin/python -r requirements.txt ipykernel`, затем выберите `.venv` как ядро ноутбука. Веса положите в `checkpoints/` (эта папка исключена из Git); пути к двум основным файлам задаются ниже. При `CLIP1_PATH = CLIP2_PATH = None` текстовые энкодеры загружаются с Hugging Face; для автономной работы задайте их локальные пути в первой ячейке. Ноутбук не загружает веса до запуска соответствующей ячейки.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'SUPIR').is_dir(), 'Запустите ноутбук из корня репозитория или notebooks/'
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
from PIL import Image
from omegaconf import OmegaConf
import CKPT_PTH

CLIP1_PATH = None  # или путь к локальному SDXL CLIP-1
CLIP2_PATH = None  # или путь к локальному SDXL CLIP-2
CKPT_PTH.SDXL_CLIP1_PATH = CLIP1_PATH
CKPT_PTH.SDXL_CLIP2_CKPT_PTH = CLIP2_PATH
from SUPIR.util import create_SUPIR_model, PIL2Tensor, Tensor2PIL, convert_dtype

assert torch.cuda.is_available(), 'SUPIR требует CUDA; запустите ноутбук на Linux-машине с NVIDIA GPU'
DEVICE = 'cuda:0'

## 1. Веса и модель

Укажите файлы SDXL Base 1.0 (вариант с 0.9 VAE из инструкции SUPIR) и SUPIR-v0Q. `create_SUPIR_model` читает пути из YAML, поэтому создаём временную копию конфигурации с локальными путями, не меняя исходный `options/SUPIR_v0.yaml`.

In [ ]:
SDXL_CKPT = ROOT / 'checkpoints/sd_xl_base_1.0_0.9vae.safetensors'
SUPIR_CKPT = ROOT / 'checkpoints/SUPIR-v0Q.ckpt'
for path in (SDXL_CKPT, SUPIR_CKPT):
    if not path.is_file():
        raise FileNotFoundError(f'Не найден файл весов: {path}')

config = OmegaConf.load(ROOT / 'options/SUPIR_v0.yaml')
config.SDXL_CKPT = str(SDXL_CKPT)
config.SUPIR_CKPT = None
config.SUPIR_CKPT_Q = str(SUPIR_CKPT)
with tempfile.TemporaryDirectory() as tmpdir:
    config_path = Path(tmpdir) / 'supir_notebook.yaml'
    OmegaConf.save(config, config_path)
    model = create_SUPIR_model(str(config_path), SUPIR_sign='Q')

model.ae_dtype = convert_dtype('bf16')
model.model.dtype = convert_dtype('fp16')
model = model.to(DEVICE).eval()

## 2. Входное изображение и восстановление

Положите RGB-изображение в `examples/input.png` или поменяйте путь. `PIL2Tensor` приводит размер к кратному 64 и переводит пиксели в диапазон `[-1, 1]`. Итог сохраняется в `results/`. Значение `restoration_scale=-1` повторяет стандартный запуск `test.py` и отключает дополнительное притяжение к LQ-латенту; для эксперимента с restoration-guided sampling можно задать `4.0`.

In [ ]:
INPUT_IMAGE = ROOT / 'examples/input.png'
if not INPUT_IMAGE.is_file():
    raise FileNotFoundError(f'Укажите существующее изображение: {INPUT_IMAGE}')

image = Image.open(INPUT_IMAGE).convert('RGB')
lq, original_h, original_w = PIL2Tensor(image)
lq = lq.unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    samples = model.batchify_sample(
        lq, [''], num_steps=50, restoration_scale=-1,
        s_churn=5, s_noise=1.01, cfg_scale=4.0, control_scale=1.0,
        seed=1234, num_samples=1, color_fix_type='Wavelet',
        use_linear_CFG=True, cfg_scale_start=1.0,
    )

result = Tensor2PIL(samples[0], original_h, original_w)
output_path = ROOT / 'results/supir_baseline.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
result.save(output_path)
print(output_path)
display(result)